# 頚椎学習曲線 — プロット

`01_train.ipynb` で収集したJSONを集計し、学習曲線をプロットする。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 設定 =====
DRIVE_LC_DIR = '/content/drive/MyDrive/spine_data/omuro_cervical_lc'
VARIANTS = ['smallunet_aug1_awl_s15']  # プロットするバリアント名
# ================

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

def load_results(variant):
    """バリアントの全JSON結果を読み込み、size -> list[dict] に整理する。"""
    results_dir = os.path.join(DRIVE_LC_DIR, 'results', variant)
    by_size = {}
    if not os.path.isdir(results_dir):
        print(f'Not found: {results_dir}')
        return by_size
    for fname in sorted(os.listdir(results_dir)):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(results_dir, fname)) as f:
            d = json.load(f)
        sz = d['size']
        by_size.setdefault(sz, []).append(d)
    return by_size

def mean_sd(vals):
    if not vals: return None, None
    m = sum(vals) / len(vals)
    sd = (sum((v-m)**2 for v in vals) / max(len(vals)-1, 1)) ** 0.5
    return m, sd

def extract_curve(by_size, key_fn):
    sizes, means, sds = [], [], []
    for sz in sorted(by_size):
        vals = [key_fn(d) for d in by_size[sz] if key_fn(d) is not None]
        if not vals: continue
        m, sd = mean_sd(vals)
        sizes.append(sz); means.append(m); sds.append(sd)
    return sizes, means, sds

print('Setup complete')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('Overall MRE (mm)',        lambda d: d['overall']['mre_mm'],                    axes[0, 0]),
    ('Overall MRE excl. outliers (mm)', lambda d: d['overall_excl_outliers']['mre_mm'], axes[0, 1]),
    ('C2C7_angle MAE (°)',      lambda d: d['overall']['angle_mae'].get('C2C7_angle'), axes[1, 0]),
    ('T1S MAE (°)',             lambda d: d['overall']['angle_mae'].get('T1S'),        axes[1, 1]),
]

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for variant, color in zip(VARIANTS, colors):
    by_size = load_results(variant)
    if not by_size:
        continue
    for title, key_fn, ax in metrics:
        sizes, means, sds = extract_curve(by_size, key_fn)
        if not sizes: continue
        ax.errorbar(sizes, means, yerr=sds, marker='o', capsize=4,
                    label=variant, color=color)
        ax.set_xlabel('Training size (cases)')
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
out_path = os.path.join(DRIVE_LC_DIR, 'learning_curve.png')
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# Per-landmark MRE カーブ
LANDMARK_ORDER = ['C2_center','C2_ant','C2_post','C7_sup_post','C7_inf_ant','C7_inf_post','T1_ant','T1_post']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for variant, color in zip(VARIANTS, colors):
    by_size = load_results(variant)
    if not by_size: continue
    for ax, lm in zip(axes, LANDMARK_ORDER):
        sizes, means, sds = extract_curve(by_size, lambda d, k=lm: d['landmarks'].get(k, {}).get('mre_mm'))
        if not sizes: continue
        ax.errorbar(sizes, means, yerr=sds, marker='o', capsize=4, label=variant, color=color)
        ax.set_title(lm, fontsize=9)
        ax.set_xlabel('N train')
        ax.set_ylabel('MRE (mm)')
        ax.axhline(2.0, color='green', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.axhline(4.0, color='orange', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.grid(alpha=0.3)

plt.tight_layout()
out_path = os.path.join(DRIVE_LC_DIR, 'learning_curve_per_landmark.png')
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')